In [1]:
modelsHyperParameters = [
    # ANN configurations
    Dict("estimator" => :ANN, "topology" => (64,), "maxEpochs" => 200, "learningRate" => 0.01),
    Dict("estimator" => :ANN, "topology" => (128,), "maxEpochs" => 150, "learningRate" => 0.005),
    Dict("estimator" => :ANN, "topology" => (64, 32), "maxEpochs" => 100, "learningRate" => 0.01),
    Dict("estimator" => :ANN, "topology" => (128, 64), "maxEpochs" => 200, "learningRate" => 0.001),
    Dict("estimator" => :ANN, "topology" => (256,), "maxEpochs" => 300, "learningRate" => 0.0005),
    Dict("estimator" => :ANN, "topology" => (128, 64, 32), "maxEpochs" => 200, "learningRate" => 0.01),
    Dict("estimator" => :ANN, "topology" => (64, 64), "maxEpochs" => 250, "learningRate" => 0.005),
    Dict("estimator" => :ANN, "topology" => (256, 128), "maxEpochs" => 200, "learningRate" => 0.001),

    # SVM configurations
    Dict("estimator" => :SVM, "kernel" => "rbf", "C" => 1.0),
    Dict("estimator" => :SVM, "kernel" => "linear"),
    Dict("estimator" => :SVM, "kernel" => "poly", "degree" => 2),
    Dict("estimator" => :SVM, "kernel" => "sigmoid", "C" => 1.0),
    Dict("estimator" => :SVM, "kernel" => "rbf", "C" => 0.01),
    Dict("estimator" => :SVM, "kernel" => "poly", "degree" => 3),
    Dict("estimator" => :SVM, "kernel" => "linear"),
    Dict("estimator" => :SVM, "kernel" => "sigmoid", "C" => 10.0),

    # Decision Tree configurations
    Dict("estimator" => :DecisionTree, "max_depth" => 3, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 5, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 7, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 10, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 15, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 20, "random_state" => 42),

    # kNN configurations
    Dict("estimator" => :KNN, "k" => 3),
    Dict("estimator" => :KNN, "k" => 5),
    Dict("estimator" => :KNN, "k" => 7),
    Dict("estimator" => :KNN, "k" => 9),
    Dict("estimator" => :KNN, "k" => 11),
    Dict("estimator" => :KNN, "k" => 15)
]

println(modelsHyperParameters)

Dict{String, Any}[Dict("maxEpochs" => 200, "learningRate" => 0.01, "estimator" => :ANN, "topology" => (64,)), Dict("maxEpochs" => 150, "learningRate" => 0.005, "estimator" => :ANN, "topology" => (128,)), Dict("maxEpochs" => 100, "learningRate" => 0.01, "estimator" => :ANN, "topology" => (64, 32)), Dict("maxEpochs" => 200, "learningRate" => 0.001, "estimator" => :ANN, "topology" => (128, 64)), Dict("maxEpochs" => 300, "learningRate" => 0.0005, "estimator" => :ANN, "topology" => (256,)), Dict("maxEpochs" => 200, "learningRate" => 0.01, "estimator" => :ANN, "topology" => (128, 64, 32)), Dict("maxEpochs" => 250, "learningRate" => 0.005, "estimator" => :ANN, "topology" => (64, 64)), Dict("maxEpochs" => 200, "learningRate" => 0.001, "estimator" => :ANN, "topology" => (256, 128)), Dict("estimator" => :SVM, "C" => 1.0, "kernel" => "rbf"), Dict("estimator" => :SVM, "kernel" => "linear"), Dict("estimator" => :SVM, "kernel" => "poly", "degree" => 2), Dict("estimator" => :SVM, "C" => 1.0, "kernel"

In [4]:
include("utils/utils.jl")

    Updating registry at `C:\Users\yanir\.julia\registries\General.toml`
   Resolving package versions...
  No Changes to `C:\Users\yanir\.julia\environments\v1.10\Project.toml`
  No Changes to `C:\Users\yanir\.julia\environments\v1.10\Manifest.toml`


oneHotEncoding (generic function with 3 methods)

In [7]:
function genModel(modelsHyperParameters:: Dict{String})
    model = nothing

    if modelsHyperParameters["estimator"] == :SVM
        model = SVC(kernel=modelsHyperParameters["kernel"],
        degree = modelsHyperParameters["degree"],
        C = modelsHyperParameters["C"])

    elseif modelsHyperParameters["estimator"] == :DecisionTree
        model = DecisionTreeClassifier(max_depth = modelsHyperParameters["max_depth"],
        random_state = modelsHyperParameters["random_state"])

    elseif modelsHyperParameters["estimator"] == :KNN
        model = KNeighborsClassifier(n_neighbors = modelsHyperParameters["k"])
        
    elseif modelsHyperParameters["estimator"] == :ANN
        model = MLPClassifier(hidden_layer_sizes = modelsHyperParameters["topology"],
        max_iter = modelsHyperParameters["maxEpochs"],
        learning_rate_init = modelsHyperParameters["learningRate"])
    end
    return model
end

genModel (generic function with 1 method)

In [ ]:
function trainClassEnsemble(modelsHyperParameters::AbstractArray{Dict{String, <:Any},1},
    inputs::AbstractArray{<:Real,2},
    targets::AbstractArray{Bool,2},
    crossvalidation::Bool=false)

    numModels = length(modelsHyperParameters)
    models = Array{Any}(undef, numModels)
    modelName = Dict(
        modelsHyperParameters["estimator"] == :ANN => "ANN",
        modelsHyperParameters["estimator"] == :SVM => "SVM",
        modelsHyperParameters["estimator"] == :DecisionTree => "DT",
        modelsHyperParameters["estimator"] == :KNN => "KNN",
        )

    if crossvalidation
        kFoldIndices = crossvalidation(size(targets,1), 10)
        numFolds =length(unique(kFoldIndices))
    else
        numFolds = 1
    end

    for fold in 1:numFolds
        trainingInputs = inputs[kFoldIndices.!=numFold, :]
        testInputs = inputs[kFoldIndices.==numFold, :]
        trainingTargets = targets[kFoldIndices.!=numFold, :]
        testTargets = targets[kFoldIndices.==numFold, :]

        for (index, modelsHyperParameters) in enumerate(modelsHyperParameters)
            
            model = genModel(modelsHyperParameters[index])

            fit!(model, trainingInputs, trainingTargets)
            acc[index] = score(model,test_input, test_output) * 100
            models[index] = deepcopy(model)
        end
    end

    else

    end
end



trainClassEnsemble (generic function with 2 methods)